<a href="https://colab.research.google.com/github/Benitmulindwa/Cheminformatics/blob/main/QSAR_1(AD).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install rdkit pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 34.4 MB/s eta 0:00:00


In [2]:
import pandas as pd

In [3]:
data = pd.read_csv("https://raw.githubusercontent.com/PatWalters/datafiles/refs/heads/main/carbonic.csv")


In [21]:
df=pd.DataFrame(data)
# df=df.rename(columns={"pIC50":"activity"})
df

,SMILES,Name,pIC50
0,CC(C)CN(C)Cc1cc(ccc1O)C(=O)c2cc(sc2)S(=O)(=O)N,CHEMBL544127,8.05
1,COc1ccc(cc1)C(=O)c2cc(oc2)S(=O)(=O)N,CHEMBL340519,8.70
2,CN(C)Cc1cc(ccc1O)C(=O)c2cc(sc2)S(=O)(=O)N,CHEMBL541089,8.19
3,c1cc(ccc1CN2CCOCC2)S(=O)(=O)c3cc(sc3)S(=O)(=O)N,CHEMBL555153,8.12
4,CC(C)CN(C)Cc1cc(ccc1O)S(=O)(=O)c2cc(sc2)S(=O)(...,CHEMBL539817,8.30
...,...,...,...
551,CN(C)C(=O)NC1CCc2c(cccc2OC)C1,CHEMBL3786651,7.76
552,CN(C)C(=O)NC1CCc2ccc(cc2C1)O,CHEMBL3785207,7.22
553,CN(C)S(=O)(=O)NC1CCc2c(cccc2O)C1,CHEMBL3786873,7.81
554,CN(C)S(=O)(=O)NC1CCc2cc(ccc2C1)O,CHEMBL3786442,7.89


In [5]:
from rdkit.Chem import AllChem, rdFingerprintGenerator
import numpy as np
from rdkit import DataStructs

In [60]:
def morgan_fingerprint(mol):
  mol=AllChem.MolFromSmiles(mol)
  if mol is None:
    return None
  generator=rdFingerprintGenerator.GetMorganGenerator(2,fpSize=2048)
  fp=generator.GetFingerprint(mol)
  arr=np.array(fp)
  # DataStructs.ConvertToNumpyArray(fp,arr)
  return arr

In [61]:
# Apply the function to create the X matrix
X = np.array([morgan_fingerprint(s) for s in df['SMILES']])
y = np.array(df['pIC50'])

In [44]:
from sklearn.model_selection import train_test_split

In [70]:
mols = [AllChem.MolFromSmiles(s) for s in df["SMILES"]]
generator=rdFingerprintGenerator.GetMorganGenerator(2, fpSize=2048)

fps_rdkit = [
    generator.GetFingerprint(m)
    for m in mols
]

In [71]:
# 3. SPLIT DATA
X_train, X_test, y_train, y_test, fps_train, fps_test = train_test_split(
    X, y, fps_rdkit, test_size=0.2, random_state=42
)

In [46]:
from sklearn.ensemble import RandomForestRegressor

In [72]:
# 4. BUILD and TRAIN MODEL
# (Using Random Forest)
rf_model=RandomForestRegressor(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

RandomForestRegressor(random_state=42)

In [73]:
from sklearn.metrics import mean_squared_error, r2_score

In [74]:
# 5.EVALUATION

y_pred=rf_model.predict(X_test)
np.sqrt(mean_squared_error(y_test,y_pred))

np.float64(0.6789331069746012)

In [75]:
print(f"SCORE: {r2_score(y_test,y_pred):.4f}")

SCORE: 0.7415


In [108]:
# Example Prediction
new_mol = "CCCCCCSCS(=O)(=O)N" # Paracetamol
new_fp = morgan_fingerprint(new_mol).reshape(1, -1)
prediction = rf_model.predict(new_fp)
print(f"\nPredicted Activity: {prediction[0]:.2f}")


Predicted Activity: 6.31


# APPLICABILITY DOMAIN(AD)

> *When should I trust my prediction?*



## **Method 1:** Similarity Based AD

Idea:


>**A molecule is inside AD if it is similar to at least one training molecule.**


Rule:

```max(Tanimoto similarity)≥threshold```

Typical threshold:

0.6 (Morgan FP, radius=2)

In [109]:
fp_new_rdkit = AllChem.GetMorganFingerprintAsBitVect(
    AllChem.MolFromSmiles(new_mol), 2, nBits=2048
)

[23:20:23] DEPRECATION WARNING: please use MorganGenerator


In [110]:
from rdkit import DataStructs

def in_ad(fp_query, fps_train, threshold=0.6):
    sims = DataStructs.BulkTanimotoSimilarity(fp_query, fps_train)
    return max(sims) >= threshold, max(sims)


In [111]:
flag, sim = in_ad(fp_new_rdkit, fps_train)
print(flag, sim)

True 1.0
